In [64]:
from dotenv import load_dotenv
import os

load_dotenv('.env')

True

In [65]:
!pip install langchain_core requests huggingface_hub langchain-huggingface


In [66]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [67]:
# creating tool

@tool
def multiply(a: int, b: int) -> int:
  """Multiply of two numbers"""
  return a*b

In [68]:
# tool binding process below:

In [69]:
!pip install langchain-openai

In [124]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

repo_id = "Qwen/Qwen2.5-7B-Instruct"
model = HuggingFaceEndpoint(repo_id=repo_id, temperature=0)
llm = ChatHuggingFace(llm=model)

In [108]:
llm_with_tools = llm.bind_tools([multiply])

In [109]:
result = llm_with_tools.invoke("Hi, can you multiply 1 with 5")

In [110]:
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":1,"b":5}', 'name': 'multiply', 'description': None}, 'id': 'call_abo1y0dy3fx2p2sf2mcycvue', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 213, 'total_tokens': 238}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b5fe2-ffc9-7482-97ec-3ea9351be208-0', tool_calls=[{'name': 'multiply', 'args': {'a': 1, 'b': 5}, 'id': 'call_abo1y0dy3fx2p2sf2mcycvue', 'type': 'tool_call'}], usage_metadata={'input_tokens': 213, 'output_tokens': 25, 'total_tokens': 238})

In [111]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'a': 1, 'b': 5},
 'id': 'call_abo1y0dy3fx2p2sf2mcycvue',
 'type': 'tool_call'}

In [112]:
result.tool_calls[0]['args']

{'a': 1, 'b': 5}

In [113]:
multiply.invoke(result.tool_calls[0]['args'])

5

In [114]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='5', name='multiply', tool_call_id='call_abo1y0dy3fx2p2sf2mcycvue')

In [115]:
from langchain_core.messages import HumanMessage

query = HumanMessage("Can u multiply 4 with 4")

messages = [query]


In [116]:
result = llm_with_tools.invoke(messages)

In [117]:
messages.append(result)

In [118]:
messages

[HumanMessage(content='Can u multiply 4 with 4', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":4,"b":4}', 'name': 'multiply', 'description': None}, 'id': 'call_4e8p8j0l2ezkfkph9xnjpvnc', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 211, 'total_tokens': 236}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b5fe3-0677-7543-9062-935f58d31d29-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 4}, 'id': 'call_4e8p8j0l2ezkfkph9xnjpvnc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 211, 'output_tokens': 25, 'total_tokens': 236})]

In [119]:
tool_result = multiply.invoke(result.tool_calls[0])

In [120]:
messages.append(tool_result)

In [121]:
messages

[HumanMessage(content='Can u multiply 4 with 4', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":4,"b":4}', 'name': 'multiply', 'description': None}, 'id': 'call_4e8p8j0l2ezkfkph9xnjpvnc', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 211, 'total_tokens': 236}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b5fe3-0677-7543-9062-935f58d31d29-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 4}, 'id': 'call_4e8p8j0l2ezkfkph9xnjpvnc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 211, 'output_tokens': 25, 'total_tokens': 236}),
 ToolMessage(content='16', name='multiply', tool_call_id='call_4e8p8j0l2ezkfkph9xnjpvnc')]

In [122]:
llm_with_tools.invoke(messages)

AIMessage(content='Sure, 4 multiplied by 4 equals 16.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 83, 'total_tokens': 97}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b5fe3-2394-77c3-a089-85354fca7006-0', usage_metadata={'input_tokens': 83, 'output_tokens': 14, 'total_tokens': 97})

In [123]:
llm_with_tools.invoke(messages).content

'Sure, 4 multiplied by 4 equals 16.'

In [195]:
# Currency Converter
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """Get the conversion factor between two currencies"""

  url = f'https://v6.exchangerate-api.com/v6/d30618068854ac147a89b4ca/pair/{base_currency}/{target_currency}'
  response = requests.get(url)
  return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """Convert one currency to another"""
  return base_currency_value * conversion_rate

In [196]:
get_conversion_factor.invoke({'base_currency':'INR','target_currency':'USD'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1766793601,
 'time_last_update_utc': 'Sat, 27 Dec 2025 00:00:01 +0000',
 'time_next_update_unix': 1766880001,
 'time_next_update_utc': 'Sun, 28 Dec 2025 00:00:01 +0000',
 'base_code': 'INR',
 'target_code': 'USD',
 'conversion_rate': 0.01111}

In [197]:
convert.invoke({'base_currency_value':20, 'conversion_rate': 0.0111})

0.222

In [198]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

repo_id = "Qwen/Qwen2.5-7B-Instruct"
model = HuggingFaceEndpoint(repo_id=repo_id, temperature=0)
llm = ChatHuggingFace(llm=model)

In [199]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [214]:
query = HumanMessage("Convert 1,110.74 USD to INR using the current exchange rate.")
messages = [query]

In [215]:
ai_messages = llm_with_tools.invoke(messages)

In [216]:
messages.append(ai_messages)

In [217]:
ai_messages.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_991a3wqjnol4tqmdbefcyvfc',
  'type': 'tool_call'}]

In [218]:
import json
for tool_call in ai_messages.tool_calls:
  # Execute the first tool and get the value of conversion rate

  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    print(tool_message1)

    # fetch the conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']

    #append this tool message to messages list
    messages.append(tool_message1)

  # Execute the second tool after conversion rate is calculated from first tool

  if tool_call['name'] == 'convert':
    # fetch the current argument

    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)


content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1766793601, "time_last_update_utc": "Sat, 27 Dec 2025 00:00:01 +0000", "time_next_update_unix": 1766880001, "time_next_update_utc": "Sun, 28 Dec 2025 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 90.0185}' name='get_conversion_factor' tool_call_id='call_991a3wqjnol4tqmdbefcyvfc'


In [219]:
messages

[HumanMessage(content='Convert 1,110.74 USD to INR using the current exchange rate.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_991a3wqjnol4tqmdbefcyvfc', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 316, 'total_tokens': 346}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b601a-44b4-70b2-a901-68806c9cb8a8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_991a3wqjnol4tqmdbefcyvfc', 'type': 'tool_call'}], usage_metadata={'input_tokens': 316, 'output_tokens': 30, 'total_tokens': 346}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.

In [220]:
llm_with_tools.invoke(messages).content

'The current exchange rate from USD to INR is 1 USD = 90.0185 INR. \n\nTo convert 1,110.74 USD to INR, we multiply 1,110.74 by 90.0185:\n\n\\[ 1,110.74 \\times 90.0185 = 100,000.00 \\text{ INR} \\]\n\nTherefore, 1,110.74 USD is approximately 100,000.00 INR.'